# T1 — Camera degradation health score (Colab, khoảng 1 giờ)
**Bối cảnh:** camera RGB của xe ADAS. Không huấn luyện CNN.

**Tái hiện một phần** [Aher, arXiv:2605.05439v1](https://arxiv.org/html/2605.05439v1):
- III-A / IV-F: công thức GSHI và effective weights trong Table I.
- IV: ý tưởng sinh degradation với severity kiểm soát, dùng phép biến đổi **giản lược**, không khẳng định giống simulator tác giả.
- VI: severity sweep và YOLOv8n cố định; dùng proxy detection consistency vì không có ground truth.

**Không tái hiện:** EfficientNet-B2, training, spatial uncertainty, health MAE=0.064, mAP hay early-warning lead time của paper.

## Cách chạy
1. Mở notebook trong Google Colab → Runtime → Change runtime type → T4 GPU nếu có (CPU vẫn chạy).
2. Chạy lần lượt các cell. Notebook tự lấy ảnh đường phố `bus.jpg` của Ultralytics, không cần upload.
3. Nếu chỉ muốn OpenCV, đặt `RUN_YOLO=False`. Nếu chạy YOLO, lần đầu cần internet để tải weights.
4. Cell cuối tải ZIP gồm CSV, ảnh minh họa, biểu đồ, cấu hình và bản báo cáo gợi ý.

Mặc định chỉ một ảnh để demo nhanh. Kết quả là case study, không chứng minh generalization; std giữa ảnh với n=1 không xác định (CSV giữ NaN). Có thể thêm URL ảnh đường phố vào `IMAGE_SOURCES` nếu muốn mở rộng.

GSHI dùng severity biết trước là **target từ simulator**, không phải score được dự đoán từ ảnh. Health proxy ở đây cần ảnh sạch tham chiếu, khác monitor một ảnh của paper. Các ngưỡng chỉ là lựa chọn cho lab, chưa được hiệu chuẩn cho xe thật.

In [ ]:
# Cài dependencies trong Colab. Nếu Colab yêu cầu restart, restart rồi chạy lại từ cell import.
%pip -q install "ultralytics>=8.3,<9" pandas matplotlib opencv-python-headless

In [ ]:
from pathlib import Path
import json, hashlib, zipfile, shutil, time, platform, importlib.metadata
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

RUN_YOLO = True
MAX_IMAGES = 20
MAX_SIDE = 640
LEVELS = [0.0, 0.25, 0.5, 0.75, 1.0] # clean + 4 mức degraded
MODES = ['motion_blur', 'low_light', 'sensor_noise', 'exposure_shift', 'blur_noise']
SEED = 42
CONF_THRESHOLD = 0.25
MATCH_IOU = 0.5
PROXY_WARNING = 0.6 # heuristic do nhóm chọn; không phải safety threshold
# Effective weights từ Table I của paper, giữ nguyên để tính H*.
WEIGHTS = {'motion_blur': 1.54, 'low_light': 1.30,
           'sensor_noise': 1.05, 'exposure_shift': 0.95}
ROOT = Path('/content/camera_health_lab')
INPUT = ROOT/'input'; OUT = ROOT/'results'
INPUT.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
(OUT/'examples').mkdir(exist_ok=True)
print('Output:', OUT)

## 1. Tự lấy ảnh mẫu trên mạng — không upload
Dùng ảnh `bus.jpg` từ bộ demo Ultralytics: có bus và người trên đường phố. Thử URL chính và URL GitHub dự phòng; nếu mạng lỗi thì lấy đúng ảnh demo đi kèm package đã cài. Không cần thao tác chọn file.

Đây là ảnh minh họa, không phải dữ liệu KITTI của paper. Giữ aspect ratio, cạnh dài tối đa 640. Có thể thêm nguồn vào `IMAGE_SOURCES`; notebook lưu nguồn và SHA256 để truy vết. Không suy ra độ chính xác từ một ảnh.

In [ ]:
import urllib.request, importlib.util
# Thêm URL khác ở đây nếu muốn benchmark nhiều ảnh. Hai URL trong cùng entry là fallback cùng một ảnh.
IMAGE_SOURCES = {
    'bus.jpg': [
        'https://ultralytics.com/images/bus.jpg',
        'https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg',
    ]
}
images = {}; image_provenance = {}
for name, urls in list(IMAGE_SOURCES.items())[:MAX_IMAGES]:
    im = None; source = None; payload = None
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=15) as response:
                payload = response.read(30_000_001)
            if len(payload) > 30_000_000: raise ValueError('Ảnh quá lớn')
            im = cv2.imdecode(np.frombuffer(payload,dtype=np.uint8),cv2.IMREAD_COLOR)
            if im is None: raise ValueError('URL không trả về ảnh đọc được')
            source = url; break
        except Exception as e:
            im = None
            print('Thử nguồn tiếp theo:', str(e)[:150])
    if im is None and name == 'bus.jpg':
        spec = importlib.util.find_spec('ultralytics')
        if spec and spec.origin:
            bundled = Path(spec.origin).parent/'assets'/'bus.jpg'
            if bundled.exists():
                payload = bundled.read_bytes()
                im = cv2.imdecode(np.frombuffer(payload,dtype=np.uint8),cv2.IMREAD_COLOR)
                source = 'Ultralytics packaged demo asset: bus.jpg (same online demo image)'
    if im is None:
        print('Bỏ qua ảnh không tải được:', name); continue
    (INPUT/Path(name).name).write_bytes(payload)
    h,w = im.shape[:2]; scale = min(1.,MAX_SIDE/max(h,w))
    images[name] = cv2.resize(im,(max(1,round(w*scale)),max(1,round(h*scale))),interpolation=cv2.INTER_AREA)
    image_provenance[name] = {'source':source,'sha256':hashlib.sha256(payload).hexdigest()}
assert images, 'Không tải được ảnh mẫu. Kiểm tra internet Colab rồi chạy lại; không cần upload.'
print(f'Loaded {len(images)} image(s); {len(images)*(1+len(MODES)*(len(LEVELS)-1))} benchmark samples.')
fig,axes = plt.subplots(1,len(images),figsize=(6*len(images),5),squeeze=False)
for ax,(name,im) in zip(axes.ravel(),images.items()):
    ax.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB)); ax.set_title(name); ax.axis('off')
plt.show()

## 2. Degradation, GSHI target và metric ảnh
Mapping severity là quy ước của benchmark này, **không phải mapping vật lý đã xác minh của paper**:
- Motion blur: kernel ngang dài 1–31 pixels (đã resize ảnh).
- Low light: nhân intensity từ 1.0 xuống 0.1; chưa mô phỏng photon/shot noise đầy đủ.
- Sensor noise: Gaussian noise với sigma từ 0–40/255; khác mô hình shot + read noise của paper.
- Exposure shift: overexposure nhân intensity từ 1.0 đến 3.0 và clipping.
- Blur + noise: hai severity cùng bằng s, để thử failure case của metric sharpness.

GSHI target: `H*=product((1-s_i)**effective_weight_i)`. Metric ảnh: variance of Laplacian, bright saturation ratio (gray >=250), dark ratio (gray <=20), mean brightness, Shannon entropy.

Health proxy của nhóm: `sharpness_relative × exposure_health_relative`, cần ảnh sạch. Noise có thể làm sharpness tăng và bị clip thành 1; đây là **giới hạn được cố ý kiểm tra**, không coi proxy là estimator đáng tin cậy cho noise.

In [ ]:
def seed_for(name):
    return SEED + int(hashlib.sha256(name.encode()).hexdigest()[:8],16)

def degrade(im, mode, s, seed):
    if s == 0: return im.copy()
    out = im.copy()
    if mode in ('motion_blur','blur_noise'):
        k = 1 + 2*int(round(15*s))
        kernel = np.zeros((k,k),np.float32); kernel[k//2,:]=1/k
        out = cv2.filter2D(out,-1,kernel)
    if mode == 'low_light': out = np.clip(out.astype(np.float32)*(1-.9*s),0,255).astype(np.uint8)
    if mode == 'exposure_shift': out = np.clip(out.astype(np.float32)*(1+2*s),0,255).astype(np.uint8)
    if mode in ('sensor_noise','blur_noise'):
        noise = np.random.default_rng(seed).normal(0,1,out.shape)*40*s
        out = np.clip(out.astype(np.float32)+noise,0,255).astype(np.uint8)
    return out

def gshi_target(mode,s):
    active = ['motion_blur','sensor_noise'] if mode == 'blur_noise' else [mode]
    return float(np.prod([(1-s)**WEIGHTS[m] for m in active]))

def image_metrics(im):
    gray = cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
    hist = np.bincount(gray.ravel(),minlength=256).astype(float); p=hist/hist.sum(); p=p[p>0]
    return {'laplacian_var':float(cv2.Laplacian(gray,cv2.CV_64F).var()),
            'saturation_ratio':float((gray>=250).mean()), 'dark_ratio':float((gray<=20).mean()),
            'mean_brightness':float(gray.mean()/255), 'entropy_bits':float(-(p*np.log2(p)).sum())}

def health_proxy(metrics,clean):
    sharp = min(1.,metrics['laplacian_var']/max(clean['laplacian_var'],1e-9))
    exposure = max(0.,1-metrics['dark_ratio']-metrics['saturation_ratio'])
    clean_exposure = max(0.,1-clean['dark_ratio']-clean['saturation_ratio'])
    return float(sharp*min(1.,exposure/max(clean_exposure,1e-9)))

# Kiểm tra algebra và invariants trước khi benchmark.
assert gshi_target('motion_blur',0)==1 and gshi_target('motion_blur',1)==0
assert np.isclose(gshi_target('blur_noise',.5),.5**(1.54+1.05))
assert all(np.diff([gshi_target('low_light',s) for s in LEVELS])<=0)

## 3. YOLOv8n có sẵn — tùy chọn, không training
Chỉ xét COCO classes person/bicycle/car/motorcycle/bus/truck. Cố định confidence threshold và image size.

Báo cáo số detection, mean confidence và **clean-box retention**: ghép boxes cùng class với IoU>=0.5 so với prediction ảnh sạch. Đây là consistency với clean prediction, **không phải recall ground truth**. `matched_confidence` chia tổng confidence của boxes ghép được cho số boxes sạch, boxes mất được tính 0 để giảm survivor bias. Nếu ảnh sạch không có detection, hai proxy này là NaN, không gán 0 hay 1.

In [ ]:
model = None
if RUN_YOLO:
    try:
        from ultralytics import YOLO
        import torch
        DEVICE = 0 if torch.cuda.is_available() else 'cpu'
        model = YOLO('yolov8n.pt')
        print('Detector device:', DEVICE)
    except Exception as e:
        print('Không tải được YOLO:',str(e))
        print('Vẫn chạy metric ảnh. Có thể upload yolov8n.pt và sửa đường dẫn rồi chạy lại.')

def detect(im):
    if model is None: return None,im.copy()
    result = model.predict(im,imgsz=640,conf=CONF_THRESHOLD,classes=[0,1,2,3,5,7],device=DEVICE,verbose=False)[0]
    return result.boxes.data.cpu().numpy(), result.plot()

def box_iou(a,b):
    lo=np.maximum(a[:2],b[:2]); hi=np.minimum(a[2:4],b[2:4]); inter=np.maximum(hi-lo,0).prod()
    aa=np.maximum(a[2:4]-a[:2],0).prod(); bb=np.maximum(b[2:4]-b[:2],0).prod()
    return float(inter/max(aa+bb-inter,1e-9))

def detector_metrics(boxes,clean_boxes):
    if boxes is None: return {k:np.nan for k in ['detections','mean_confidence','clean_box_retention','matched_confidence']}
    count=len(boxes)
    pairs=sorted([(box_iou(a,b),i,j) for i,a in enumerate(clean_boxes) for j,b in enumerate(boxes)
                  if int(a[5])==int(b[5])],reverse=True)
    used_i=set(); used_j=set(); confidence_sum=0.
    for iou,i,j in pairs:
        if iou<MATCH_IOU: break
        if i not in used_i and j not in used_j:
            used_i.add(i); used_j.add(j); confidence_sum+=float(boxes[j,4])
    return {'detections':count, 'mean_confidence':float(boxes[:,4].mean()) if count else np.nan,
            'clean_box_retention':len(used_i)/len(clean_boxes) if len(clean_boxes) else np.nan,
            'matched_confidence':confidence_sum/len(clean_boxes) if len(clean_boxes) else np.nan}

In [ ]:
records=[]; previews={}; started=time.perf_counter()
for idx,(name,im) in enumerate(images.items()):
    clean=image_metrics(im); clean_boxes,clean_annotated=detect(im)
    base={'image':name,'mode':'clean','severity':0.,'gshi_target':1.,'health_proxy':health_proxy(clean,clean),**clean,
          **detector_metrics(clean_boxes,clean_boxes)}
    records.append(base)
    if idx==0: previews[('clean',0.)]=clean_annotated
    for mode in MODES:
        for s in LEVELS[1:]:
            damaged=degrade(im,mode,s,seed_for(name)); metrics=image_metrics(damaged)
            boxes,annotated=detect(damaged)
            records.append({'image':name,'mode':mode,'severity':s,'gshi_target':gshi_target(mode,s),
                            'health_proxy':health_proxy(metrics,clean),**metrics,**detector_metrics(boxes,clean_boxes)})
            if idx==0: previews[(mode,s)]=annotated
    print(f'{idx+1}/{len(images)}: {name}')
df=pd.DataFrame(records)
df.to_csv(OUT/'per_image_metrics.csv',index=False)
# Chèn clean baseline cho từng đường severity sweep, nhưng raw CSV chỉ chứa clean một lần/ảnh.
sweep=pd.concat([df[df['mode']!='clean']]+[df[df['mode']=='clean'].assign(mode=m) for m in MODES],ignore_index=True)
metric_columns=['gshi_target','health_proxy','laplacian_var','saturation_ratio','dark_ratio','mean_brightness',
                'entropy_bits','detections','mean_confidence','clean_box_retention','matched_confidence']
summary=sweep.groupby(['mode','severity'])[metric_columns].agg(['mean','std','count'])
summary.columns=['_'.join(c) for c in summary.columns]; summary=summary.reset_index()
summary.to_csv(OUT/'summary.csv',index=False)
print(f'Finished in {time.perf_counter()-started:.1f}s. Không gồm thời gian cài/tải weights.')
display(summary[['mode','severity','gshi_target_mean','health_proxy_mean','laplacian_var_mean',
                 'saturation_ratio_mean','entropy_bits_mean','clean_box_retention_mean']].round(3))

## 4. Ảnh trước/sau và biểu đồ
Error bars là std giữa các ảnh, không phải confidence interval. Số valid samples cho các detector proxy có trong cột `_count` của summary CSV. Nhiều detection hơn hoặc confidence cao hơn không tự động đồng nghĩa accuracy cao hơn.

In [ ]:
fig,axes=plt.subplots(len(MODES),len(LEVELS),figsize=(18,3*len(MODES)),squeeze=False)
for r,mode in enumerate(MODES):
    for c,s in enumerate(LEVELS):
        im=previews[('clean',0.)] if s==0 else previews[(mode,s)]
        axes[r,c].imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))
        axes[r,c].set_title(f'{mode}, s={s:g}\nH*={gshi_target(mode,s):.3f}'); axes[r,c].axis('off')
        cv2.imwrite(str(OUT/'examples'/f'{mode}_{s:.2f}.png'),im)
fig.tight_layout(); fig.savefig(OUT/'before_after.png',dpi=130,bbox_inches='tight'); plt.show()

plot_metrics=['gshi_target','health_proxy','laplacian_var','saturation_ratio','entropy_bits','clean_box_retention']
fig,axes=plt.subplots(2,3,figsize=(17,9))
for ax,metric in zip(axes.ravel(),plot_metrics):
    for mode in MODES:
        sub=summary[summary['mode']==mode]
        ax.errorbar(sub['severity'],sub[metric+'_mean'],yerr=sub[metric+'_std'].fillna(0),marker='o',capsize=2,label=mode)
    ax.set_title(metric); ax.set_xlabel('Severity (quy ước simulator)'); ax.grid(alpha=.3)
axes[0,0].legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT/'metric_curves.png',dpi=150); plt.show()

## 5. Failure case và đề xuất down-weight
So sánh blur với blur+noise: xem Laplacian có tăng lại dù chi tiết ảnh đã bị blur. Health proxy có thể bỏ sót lỗi này.

Bảng dưới tìm **mức đầu tiên trong sweep** mà proxy < ngưỡng do nhóm đặt. Nó không phải kết luận camera unsafe. Không báo cáo early-warning lead time vì thiếu mAP ground truth và proxy dùng clean reference. Paper dùng các ngưỡng riêng cho GSHI, có thể hiệu chuẩn; không chuyển chúng thành luật an toàn cho xe thật.

In [ ]:
alerts=[]
for mode in MODES:
    sub=summary[summary['mode']==mode].sort_values('severity')
    hit=sub[sub['health_proxy_mean']<PROXY_WARNING]
    alerts.append({'mode':mode,'proxy_threshold':PROXY_WARNING,
                   'first_tested_severity_below_threshold':float(hit.iloc[0]['severity']) if len(hit) else np.nan})
alert_df=pd.DataFrame(alerts); alert_df.to_csv(OUT/'proxy_alerts.csv',index=False); display(alert_df)
comparison=summary[summary['mode'].isin(['motion_blur','blur_noise'])][
    ['mode','severity','laplacian_var_mean','health_proxy_mean','clean_box_retention_mean']]
display(comparison.round(3))

## 6. Xuất kết quả cho báo cáo
Checklist: ADAS; camera RGB; metric định lượng; failure case; phân biệt kết quả paper/nhóm; giải thích proxy.

**Cách trình bày trung thực:** “Nhóm tái hiện công thức health target GSHI và severity sweep trên simulator giản lược. Nhóm bổ sung baseline ảnh và detector consistency. Không tái hiện learned monitor và không khẳng định đạt các kết quả tác giả.”

Nếu còn thời gian: thử thêm ảnh ngày/đêm để kiểm tra threshold riêng. Phải chọn/calibrate threshold trên tập riêng rồi đánh giá trên ảnh khác; không tự chọn ngưỡng trên cùng tập để tuyên bố generalization. Muốn tính mAP cần nhãn bounding box thật và evaluation phù hợp.

In [ ]:
config={'paper':'https://arxiv.org/html/2605.05439v1','seed':SEED,'levels':LEVELS,'modes':MODES,
        'effective_weights':WEIGHTS,'max_side':MAX_SIDE,'run_yolo_requested':RUN_YOLO,
        'yolo_loaded':model is not None,'model':'yolov8n.pt','confidence_threshold':CONF_THRESHOLD,
        'match_iou':MATCH_IOU,'proxy_warning':PROXY_WARNING,'images':list(images),'image_provenance':image_provenance,
        'versions':{p:importlib.metadata.version(p) for p in ['numpy','pandas','matplotlib','opencv-python-headless']}}
if model is not None: config['versions']['ultralytics']=importlib.metadata.version('ultralytics')
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
report=f"""# Camera degradation health score — báo cáo benchmark
## Bài toán và phạm vi
Xe ADAS; sensor camera RGB. {len(images)} ảnh/frame, {len(LEVELS)} mức severity mỗi loại, {len(df)} samples.
Paper tham chiếu: https://arxiv.org/html/2605.05439v1 (preprint; không khẳng định đã accept).
## Phần tái hiện
Công thức GSHI target, effective weights Table I; severity sweep. Phép sinh degradation giản lược do nhóm chọn.
Không tái hiện CNN monitor, training hoặc mAP/lead time tác giả báo cáo.
## Metric
Laplacian variance, saturation ratio, dark ratio, brightness, entropy; reference-based health proxy.
YOLO loaded: {model is not None}. Detector proxies: count, mean confidence, clean-box retention, matched confidence.
Không có ground truth: detector consistency không phải precision/recall/mAP và có thể thừa hưởng lỗi clean prediction.
## Kết quả nhóm tự đo
Xem summary.csv, per_image_metrics.csv, before_after.png, metric_curves.png.
Ngưỡng health proxy: {PROXY_WARNING}, do nhóm chọn, chưa hiệu chuẩn. Xem proxy_alerts.csv.
## Nhận xét cần điền sau khi xem kết quả
- Loại degradation nào làm metric thay đổi rõ nhất? Trích số cụ thể từ CSV.
- Blur+noise có làm Laplacian/health proxy tăng so với blur không? Nếu không, báo cáo đúng dữ liệu quan sát.
- Detection có mất khi severity tăng? Chỉ mô tả consistency, không suy ra accuracy.
- Khi proxy thấp hoặc vùng quan trọng mất chi tiết, đề xuất giảm trọng số camera và kiểm tra sensor khác;
  đây là đề xuất lab, không phải chính sách safety đã xác minh.
## Giới hạn
Tập nhỏ; clean-reference proxy; simulator mappings tự chọn; không đánh giá dữ liệu thực adverse weather;
GSHI target giảm theo severity do công thức, không chứng minh monitor dự đoán được từ ảnh.
Mặc định một ảnh mẫu đường phố: case study, không đủ chứng minh generalization. Không so trực tiếp trị số với paper do khác dữ liệu/mapping/evaluation.
"""
(OUT/'report_template.md').write_text(report,encoding='utf-8')
archive=shutil.make_archive(str(ROOT/'camera_health_lab_results'),'zip',OUT)
print('Saved:',archive)
from google.colab import files
files.download(archive)